In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import  col, to_json, regexp_extract, to_date, row_number,input_file_name
from pyspark.sql.window import Window
import os
import sys
import json


In [30]:
spark = SparkSession.builder \
    .appName("BronzeStep") \
    .config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:3.5.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


In [31]:
current_path = os.getcwd()

dir_path = os.path.dirname(os.path.dirname(current_path))

with open(dir_path + '/steam/config.json', 'r') as arquivo:
  config = json.load(arquivo)

In [32]:
# Verificar o ambiente
if "dev" in dir_path:
    env = "dev"
elif "prod" in current_path:
    env = "prod"
else:
    env = "unknown"

print(f"A variável de ambiente é: {env}")

A variável de ambiente é: dev


In [33]:
path = dir_path + "/datalake/steam/user/inbound/*.json"

df = spark.read.json(path)

In [34]:
sys.path.append(dir_path + "/meus_scripts_pyspark")

from file_path import file_path
from file_name import file_name
from extract_date_from_filename import extract_date_from_filename
from column_to_date import column_to_date
from organize_files import organize_files
from convert_column_to_json import convert_column_to_json
from extract_column_json import extract_column_json
from drop_column import drop_column

In [35]:
df = file_path(df)

df = file_name(df,"file_name")

df = extract_date_from_filename(df,"file_name",r"\d{4}-\d{2}-\d{2}")

df = column_to_date(df,"file_date","yyyy-MM-dd")

df = organize_files(df,"appid","file_date")

df = convert_column_to_json(df, "package_groups")

df = extract_column_json(df,"pc_requirements_minimum","pc_requirements.minimum")
df = extract_column_json(df,"pc_requirements_recommended","pc_requirements.recommended")

df = drop_column(df,"pc_requirements")

In [36]:
df.show(truncate=False)

+-------+---------------------------------------------------------------------------------------------------------------+-------+-------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [37]:
output_path = dir_path + "/datalake/steam/user/bronze"
os.makedirs(output_path, exist_ok=True)
df.write.format('avro') \
    .mode('overwrite') \
    .partitionBy('file_date') \
    .option('overwriteSchema', 'true') \
    .save(output_path)

print(f"DataFrame salvo no formato Avro em: {output_path}")

DataFrame salvo no formato Avro em: /home/fececa/dev-env/datalake/steam/user/bronze
